In [1]:
# import libraries

# for data and function imports
from dotenv import load_dotenv
import json
import os
import sys
from pathlib import Path

# for language models
from openai import AsyncOpenAI
import gensim.downloader as api

# for concurrency handling
import asyncio

# set path to project root and import custom classes and functions
base_path = Path.cwd() / "../"
sys.path.append(str(base_path.resolve()))

# import custom functions
from utils.data_augmentation import augmentation_non_entity, augmentation_entity, find_entity_span

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/maxweiland/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# load the data
with open("../01_data/classification/annotations/annotations_no_augmentations.json", "r") as f:
    data_non_augmented = json.load(f)

In [3]:
# Function for generative paraphrasing
async def create_augmentation(client, task, idx, temp, gpt_model, glove_model):

    # extract the sentence and all annotations
    sentence = task["sentence"]
    annotations = sorted(task["annotations"], key=lambda x: x["start"])

    # set the task id
    task_id = f"index_{idx+1}"

    # apply synonym augmentations
    non_entity_aug = augmentation_non_entity(task, glove_model, aug_p=0.4)
    entity_aug = augmentation_entity(task, glove_model, aug_p_entity=1, aug_p_non_entitiy=0.4)

    # construct the prompt for the generative model
    if not annotations:
        prompt = f"""
        You are given the following sentence:
        Sentence: {sentence}

        Paraphrase the sentence in natural English while preserving its meaning.
        Return only the paraphrased sentence.
        """
    else:
        # Construct a prompt that preserves entities
        prompt = f"""
        You are given the following sentence with annotations for social groups:
        Sentence: {sentence}
        Social groups: {annotations}

        Paraphrase the sentence in natural English. while preserving the social groups exactly as they appear.
        Return only the paraphrased sentence.
        """

    # call the model, get the response and find the annotations within it
    response = await client.chat.completions.create(
        model=gpt_model,
        messages=[{"role": "user", "content": prompt}],
        #temperature=temp
    )
    paraphrased_sentence = response.choices[0].message.content.strip()
    generative_aug = find_entity_span(task, paraphrased_sentence)

    # compile the new task
    new_task = {"id": task_id,
                "sentence": sentence,
                "annotations": annotations,
                "augmentations": [
                    non_entity_aug,
                    entity_aug,
                    generative_aug
                    ]}
    
    return new_task

async def dispatch_all(client, gpt_model, glove_model, data, temp, safe_interval):
    tasks = []
    for idx, task in enumerate(data):
        # create a task and fire it, do not wait
        task = asyncio.create_task(create_augmentation(client, task, idx, temp, gpt_model, glove_model))
        tasks.append(task)
        
        # wait before starting the next request
        await asyncio.sleep(safe_interval)

    # gather all results once everything is started
    return await asyncio.gather(*tasks)

In [11]:
# load environment and create client with api key
load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# load the glove model and set which gpt model
glove_model = api.load("glove-wiki-gigaword-50")
gpt_model = "gpt-5-nano"

# set a safe rpm and calculate the interval
safe_rpm = 150
safe_interval = 60.0/safe_rpm

dataset_augmentations = await dispatch_all(client=client, gpt_model=gpt_model, glove_model=glove_model, data=data_non_augmented, temp=0.5, safe_interval=safe_interval)

In [66]:
# investigate if augmentations have been created correctly

# check for mistake in paraphrases
wrong_indices = []
for idx, item in enumerate(dataset_augmentations):
    num_original_annotations = len(item["annotations"])
    num_paraphrased_annotations = len(item["augmentations"][-1]["annotations"])
    if num_original_annotations != num_paraphrased_annotations:
        wrong_indices.append(idx)

print(wrong_indices)

[]


In [67]:
with open("../01_data/classification/annotations/annotations_augmentations.json", "w") as f:
    json.dump(dataset_augmentations, f)